# boolean-mask-identity-replace — ex6: causal attention mask — build, apply, visualize

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-identity-replace`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Mask & substitute — quick refresher

**Build a mask.** Any comparison returns a `dtype=bool` tensor of the same shape: `x < 0`, `x.abs() < eps`, `(x > 0) & (x < 1)`. Combine with `&`, `|`, `~`.

**Write through a mask.** `y[mask] = value` modifies in place. Scalars broadcast; tensor values must match the shape of `y[mask]` after broadcasting. Always `clone()` first if the function must not mutate its input.

**The dangerous case.** When a mask is the *wrong* shape, indexing can silently collapse axes or pick the wrong cells. Always check `mask.sum()` and `mask.shape` before trusting the result.

### Exercise 6 — causal attention mask — build, apply, visualize

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Construct a causal attention mask via index comparison, apply it to scores with -inf, and visualize both.
> Keywords: causal-mask, attention, lower-triangular, imshow, -inf-fill
> ```

**KCs targeted:** `mask-from-comparison`, `masked-fill-neg-inf`, `broadcast-mask-over-batch`

Implement `ex6_causal_mask_apply(scores)` to (1) build a causal (lower-triangular) boolean mask for sequence length `T = scores.shape[-1]`, then (2) return a copy of `scores` with the masked-out (strictly upper-triangular) positions set to `-inf`.

Build the mask via index comparison (not `torch.tril` — we want you to see the broadcast):
- `idx = torch.arange(T)`  → shape `(T,)`
- `mask = idx[None, :] <= idx[:, None]`  → shape `(T, T)`, lower-triangular, True on/below the diagonal

Then `scores.masked_fill(~mask, float('-inf'))` (or `scores[~mask] = -inf` on a clone — but masked_fill is cleaner because it broadcasts to the leading batch axes for free).

After the function passes, the test cell plots two heatmaps side-by-side: the boolean mask, and a softmax of the masked scores — you should see a clean lower-triangular structure in both.

In [ ]:
def ex6_causal_mask_apply(scores: Tensor) -> Tensor:
    """Apply causal mask to attention scores of shape (..., T, T).

    Returns a tensor of the same shape with upper-triangular positions set to -inf.
    Build the mask via arange + broadcast comparison; do NOT use torch.tril.
    """
    raise NotImplementedError()


def _test_ex6():
    # Unbatched
    T = 5
    scores = t.randn(T, T)
    masked = ex6_causal_mask_apply(scores)
    assert masked.shape == (T, T)
    for i in range(T):
        for j in range(T):
            if j > i:
                assert masked[i, j].item() == float('-inf'), f'expected -inf at ({i},{j}) got {masked[i, j].item()}'
            else:
                assert masked[i, j].item() == scores[i, j].item(), f'value changed at ({i},{j})'

    # Batched — mask broadcasts over leading axes
    B, H, T = 2, 3, 4
    batched = t.randn(B, H, T, T)
    out = ex6_causal_mask_apply(batched)
    assert out.shape == (B, H, T, T)
    # upper-triangular positions in EVERY batch slice must be -inf
    for b in range(B):
        for h in range(H):
            for i in range(T):
                for j in range(i + 1, T):
                    assert out[b, h, i, j].item() == float('-inf')
            # diagonal + below preserved
            for i in range(T):
                for j in range(i + 1):
                    assert out[b, h, i, j].item() == batched[b, h, i, j].item()

    # Softmax check: after masking, each row's softmax must sum to 1 and be zero on the upper-tri
    T = 6
    raw = t.randn(T, T)
    m = ex6_causal_mask_apply(raw)
    probs = m.softmax(dim=-1)
    assert t.allclose(probs.sum(dim=-1), t.ones(T), atol=1e-5), 'softmax rows must sum to 1'
    for i in range(T):
        for j in range(i + 1, T):
            assert probs[i, j].item() == 0.0, f'masked position ({i},{j}) should be 0 after softmax'

    # Visualize: build the mask + plot the softmax of masked scores
    import matplotlib.pyplot as plt
    T_vis = 10
    raw_vis = t.randn(T_vis, T_vis)
    idx = t.arange(T_vis)
    mask_vis = (idx[None, :] <= idx[:, None])
    probs_vis = ex6_causal_mask_apply(raw_vis).softmax(dim=-1)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(mask_vis.numpy(), cmap='gray_r')
    axes[0].set_title('causal mask  (True = keep)')
    im = axes[1].imshow(probs_vis.numpy(), cmap='viridis')
    axes[1].set_title('softmax(masked scores)')
    plt.colorbar(im, ax=axes[1])
    for ax in axes:
        ax.set_xlabel('key position'); ax.set_ylabel('query position')
    plt.tight_layout(); plt.show()
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_causal_mask_apply(scores: Tensor) -> Tensor:
    T = scores.shape[-1]
    idx = t.arange(T, device=scores.device)
    keep = idx[None, :] <= idx[:, None]   # (T, T), True on/below diagonal
    return scores.masked_fill(~keep, float('-inf'))
```

**Why mask BEFORE softmax.** `softmax(-inf) = 0` exactly, so the masked positions get zero probability *and* the remaining unmasked positions still sum to 1 (the `exp(-inf) = 0` terms drop out of the denominator). If you instead masked AFTER softmax (multiplying by the bool mask), each row would no longer sum to 1.

**Why arange-broadcast and not `torch.tril(torch.ones(...))`.** Both work, but the arange-compare form generalizes — swap `<=` for `< k` and you get a *banded* mask, swap `idx` for token-position IDs and you get a per-token-aware mask (useful when packing multiple sequences into one row).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()